# Amazon Price Detector

This notebook demonstrates the deployment of an open-source model through an interactive web interface.
I am utilising:
*   **QLoRA:** The architecture supports merging low-rank adapters over quantized weights.
*   **Model Selection:** Loading an open base model (TinyLlama-1.1B) in 4-bit precision.
*   **User Interface:** Wrapping the resulting AI pipeline in a visual Gradio Block implementation.

In [ ]:
import re
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading Base Model: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Model loaded successfully!")

In [ ]:
def predict_price(description: str) -> str:
    """
    Takes a product description, formats the prompt, and extracts numbers.
    """
    if not description.strip():
        return "Please enter a product description."
        
    prompt_template = f"<|system|>You are an expert pricing algorithm. Estimate the price of the following product in USD. Only output the numeric price without text or symbols.\n<|user|>{description}\n<|assistant|>Price: $"
    
    inputs = tokenizer(prompt_template, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            temperature=0.01,
            pad_token_id=tokenizer.eos_token_id
        )
        
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_response.split("Price: $")[-1].strip()
    
    match = re.search(r"(\d+(?:\.\d{2})?)", generated_text)
    
    if match:
        return f"${float(match.group(1)):.2f}"
    else:
        return f"Unknown ({generated_text[:20]}...)"


In [ ]:
with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    gr.Markdown("# 🏷️ Amazon Price Predictor")
    gr.Markdown("Week 7 Project: TinyLlama 1B local inference")
    
    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### Product Input")
            description_input = gr.Textbox(
                label="Paste Amazon description here...",
                lines=5,
                placeholder="e.g. Sony Wireless Premium Noise Canceling Headphones"
            )
            submit_btn = gr.Button("Predict Cost", variant="primary")
            
        with gr.Column(scale=1):
            gr.Markdown("### Model Output")
            price_output = gr.Textbox(
                label="Predicted Price (USD)", 
                lines=2, 
                interactive=False,
                text_align="center"
            )
            accuracy_disclaimer = gr.Markdown("*Powered by TinyLlama-1.1B via 4-bit BitsAndBytes quantization.*")

    submit_btn.click(fn=predict_price, inputs=description_input, outputs=price_output)
    description_input.submit(fn=predict_price, inputs=description_input, outputs=price_output)
    
app.launch(inline=False, inbrowser=True)
